In [1]:
!nvidia-smi

Mon Apr 20 20:20:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             25W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
import sys

# Supprimer physiquement les anciennes versions
!find /usr/local/lib/python3.12/dist-packages -maxdepth 1 \
    -name "numpy*" -o -name "scikit_learn*" | xargs rm -rf

# Réinstaller proprement
!pip install "numpy==1.26.4" "scikit-learn==1.4.2" \
    --no-cache-dir --force-reinstall

In [3]:
%%capture
import subprocess

# Supprimer TOUS les résidus NumPy et sklearn physiquement
subprocess.run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 \
    \( -name 'numpy*' -o -name 'scikit*' -o -name 'sklearn*' \) \
    -exec rm -rf {} +", shell=True)

# Vérifier que c'est bien vide
result = subprocess.run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'numpy*'",
                        shell=True, capture_output=True, text=True)
print("Résidus numpy restants :", result.stdout or "aucun ✓")

In [4]:
%%capture
!pip install numpy==1.26.4 --no-cache-dir
!pip install scikit-learn==1.4.2 --no-cache-dir

In [5]:
%%capture
# Purge totale
!find /usr/local/lib/python3.12/dist-packages -maxdepth 1 \
    \( -name 'numpy*' -o -name 'scikit*' -o -name 'sklearn*' \) \
    -exec rm -rf {} +

# NumPy 2.x + sklearn compatibles
!pip install "numpy>=2.2.0" --no-cache-dir
!pip install "scikit-learn>=1.6.0" --no-cache-dir

# PyTorch + torchcodec
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu126 --no-cache-dir
!pip install torchcodec==0.3.0 \
    --index-url=https://download.pytorch.org/whl/cu126 --no-cache-dir

In [6]:
import numpy as np
import numpy.rec  # test direct
print("NumPy :", np.__version__)  # doit être 2.2.x+

import sklearn
print("sklearn :", sklearn.__version__)

import torch
print("PyTorch :", torch.__version__)
print("CUDA :", torch.cuda.is_available())

from torchcodec.samplers import clips_at_regular_timestamps
print("torchcodec OK ✓")

from sklearn.model_selection import train_test_split
print("sklearn OK ✓")

NumPy : 2.0.2


ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [ ]:
from sklearn.model_selection import train_test_split
from torchcodec.samplers import clips_at_regular_timestamps
from timm.data.transforms_factory import create_transform
from torch.utils.data import DataLoader, Dataset
from torchcodec.decoders import VideoDecoder
from matplotlib import pyplot as plt
from torch.nn import functional as F
from tqdm.auto import tqdm
from time import time
from torch import nn
import pandas as pd
import numpy as np
import warnings
import timm
import torch
import cv2
import os

warnings.filterwarnings('ignore')

In [ ]:
CONFIG = dict(
    DEBUG = False,
    DATASET_PATH = "/kaggle/input/reencoded-barbados-traffic",
    MODEL_NAME='timm/efficientnet_b2.ra_in1k',
    PRETRAINED=True,
    NUM_CLASSES=0,
    GLOBAL_POOL='avg',
    DEVICE='cuda',
    DTYPE=torch.float16,
    BATCH_SIZE=8,
    NUM_FRAMES=128,
    DURATION=60,
    CROP_DURATION=40,
    IMAGE_SIZE=224,
    OUTPUT_DIR ='/kaggle/working/features',
    OUTPUT_FORMAT = 'parquet',
    VIDEO_FORMAT = 'mp4',
)

In [ ]:
df = pd.read_csv(os.path.join(CONFIG['DATASET_PATH'], 'Train.csv'))
df['videos'] = df['videos'].apply(lambda x: os.path.join(CONFIG['DATASET_PATH'], x))
df.head()

In [ ]:
def crop_or_pad_video(frames, max_frames):
    num_frames = frames.shape[0]
    if num_frames >= max_frames:
        return frames[:max_frames]
    else:
        pad_size = max_frames - num_frames
        pad_frames = torch.zeros((pad_size, *frames.shape[1:]), dtype=frames.dtype, device=frames.device)
        return torch.cat([frames, pad_frames], dim=0)

def apply_transform_to_video(frames: torch.Tensor, transform) -> torch.Tensor:
    """
    Applique un transform timm/torchvision en batch sur toutes les frames.
    
    Args:
        frames: Tensor [T, C, H, W] (uint8, 0-255)
        transform: Transform timm/torchvision (compatible torch tensors)
        
    Returns:
        Tensor [T, C, H, W] (float, normalisé)
    """
    # Les transforms torchvision (via timm resolve_data_config) acceptent 
    # les tensors torch directement en mode batch [B, C, H, W]
    # On traite T frames comme un batch de T images
    return transform(frames.float())
    
@torch.inference_mode()
def load_video(
    filepath, 
    device='cuda', 
    method='start',
    num_frames=128, 
    duration=60, 
    crop_duration=40, 
    image_size=224, 
    num_workers=4, 
    transform=None,
    fallback_to_cpu=True,
    ):
    decoder_kwargs = {}
    if device is None:
        device = 'cpu'
        decoder_kwargs['num_ffmpeg_threads'] = num_workers
    try:
        decoder = VideoDecoder(
            filepath, 
            device=device,
            **decoder_kwargs
            )
    except RuntimeError as e:
        decoder_kwargs['num_ffmpeg_threads'] = num_workers
        if fallback_to_cpu and device != 'cpu':
            warnings.warn(f"Falling back to CPU for video decoding for file: {filepath}")
            decoder = VideoDecoder(
                filepath, 
                device='cpu',
                **decoder_kwargs
                )
        else:
            raise RuntimeError(f"Failed to initialize VideoDecoder for file: {filepath}") from e
    
    metadata = decoder.metadata
    video_duration = getattr(metadata, "duration_seconds", 0)
    if method == 'center':
        center, half_duration = duration / 2.0, crop_duration / 2.0
        sampling_range_start=(center - half_duration),
        sampling_range_end=(center + half_duration),
    elif method == 'start':
        sampling_range_start = 0.0
        sampling_range_end = min(sampling_range_start + crop_duration, video_duration)
    elif method == 'end':
        sampling_range_end = video_duration
        sampling_range_start = max(sampling_range_end - crop_duration, 0.0)
    else:
        raise ValueError(f"Unknown method: {method}")

    decoder = clips_at_regular_timestamps(
        decoder,
        seconds_between_clip_starts=crop_duration / num_frames,
        num_frames_per_clip=1,
        sampling_range_start=sampling_range_start,
        sampling_range_end=sampling_range_end,
    )
    frames = decoder.data.squeeze(1)  # Remove the extra dimension [T, C, H, W]
    frames = crop_or_pad_video(frames, num_frames)
    
    if transform is not None:
        # Appliquer le transform en batch (T frames traitées comme batch)
        # Les frames doivent être sur CPU pour torchvision transforms
        frames = apply_transform_to_video(frames, transform)
    else:
        # Pas de transform: juste resize et normalisation basique
        frames = F.interpolate(
            frames.float(),
            size=(image_size, image_size),
            mode='bilinear',
            align_corners=False,
        )
        # Normaliser en [0, 1] et convertir en float
        frames = frames / 255.0
        
    return frames

In [ ]:
# Dataset personnalisé pour charger et prétraiter les vidéos de trafic
class TrafficDataset(Dataset):
    """
    Dataset PyTorch pour le chargement de vidéos de trafic avec multi-label et features auxiliaires.
    
    Gère:
    - Multi-label: congestion_enter_rating et congestion_exit_rating (conversion string -> int)
    - Features temporelles: time_segment_id, cycle_phase
    - Features catégorielles: signaling
    """
    
    # Colonnes de labels et features
    LABEL_COLS = ['congestion_enter_rating', 'congestion_exit_rating']
    TEMPORAL_COLS = ['time_segment_id', 'cycle_phase']
    CATEGORICAL_COLS = ['signaling']
    
    def __init__(
        self, 
        dataframe, 
        num_frames=128, 
        duration=60, 
        crop_duration=40, 
        image_size=224, 
        device='cuda', 
        transform=None,
        debug=False,
        fallback_to_cpu=True,
        label_cols=None,
        temporal_cols=None,
        categorical_cols=None,
        fit_encoders=True,
        encoders=None,
        label_encoders=None,
        temporal_stats=None,
        ):
        """
        Initialise le dataset avec les paramètres de chargement des vidéos.
        
        Args:
            dataframe: DataFrame contenant les informations des vidéos
            num_frames: Nombre de frames à extraire de chaque vidéo
            duration: Durée totale de la vidéo en secondes
            crop_duration: Durée du segment à extraire en secondes
            image_size: Taille de redimensionnement des frames
            device: Device pour le décodage ('cuda' ou 'cpu')
            transform: Transformations à appliquer aux frames
            debug: Mode debug
            fallback_to_cpu: Utilise le CPU en cas d'échec GPU
            label_cols: Liste des colonnes de labels (default: LABEL_COLS)
            temporal_cols: Liste des colonnes temporelles (default: TEMPORAL_COLS)
            categorical_cols: Liste des colonnes catégorielles (default: CATEGORICAL_COLS)
            fit_encoders: Si True, ajuste les encodeurs sur les données
            encoders: Dict d'encodeurs pré-ajustés pour features catégorielles (pour validation/test)
            label_encoders: Dict d'encodeurs pré-ajustés pour labels (pour validation/test)
            temporal_stats: Dict de statistiques temporelles pré-calculées (pour validation/test)
        """
        self.dataframe = dataframe.copy()
        self.num_frames = num_frames
        self.duration = duration
        self.crop_duration = crop_duration
        self.image_size = image_size
        self.device = device
        self.transform = transform
        self.debug = debug
        self.fallback_to_cpu = fallback_to_cpu
        
        # Configuration des colonnes
        self.label_cols = label_cols or self.LABEL_COLS
        self.temporal_cols = temporal_cols or self.TEMPORAL_COLS
        self.categorical_cols = categorical_cols or self.CATEGORICAL_COLS
        
        # Encodeurs pour les features catégorielles et labels
        self.encoders = encoders or {}
        self.label_encoders = label_encoders or {}
        self.temporal_stats = temporal_stats or {}
        
        # Setup des encodeurs
        self._setup_labels_encoders(fit_encoders)
        self._setup_features(fit_encoders)
        
    def _setup_labels_encoders(self, fit_encoders: bool):
        """
        Configure les encodeurs pour les labels multi-classes.
        Convertit les labels string en IDs numériques.
        """
        from sklearn.preprocessing import LabelEncoder
        
        self.num_classes = {}
        
        for col in self.label_cols:
            if col not in self.dataframe.columns:
                continue
            
            # Vérifier si le label est déjà numérique
            if self.dataframe[col].dtype in ['int64', 'int32', 'float64', 'float32']:
                # Déjà numérique, juste compter les classes
                self.num_classes[col] = self.dataframe[col].nunique()
                self.dataframe[f'{col}_encoded'] = self.dataframe[col].astype(int)
                if self.debug:
                    print(f"Label '{col}' is already numeric. Unique values: {sorted(self.dataframe[col].unique())}")
            else:
                # String -> besoin d'encoder
                if fit_encoders or col not in self.label_encoders:
                    encoder = LabelEncoder()
                    self.dataframe[f'{col}_encoded'] = encoder.fit_transform(
                        self.dataframe[col].fillna('unknown').astype(str)
                    )
                    self.label_encoders[col] = encoder
                    if self.debug:
                        print(f"Label '{col}' encoded: {dict(zip(encoder.classes_, range(len(encoder.classes_))))}")
                else:
                    # Utiliser l'encodeur existant (validation/test)
                    encoder = self.label_encoders[col]
                    known_classes = set(encoder.classes_)
                    self.dataframe[f'{col}_encoded'] = self.dataframe[col].fillna('unknown').astype(str).apply(
                        lambda x: encoder.transform([x])[0] if x in known_classes else -1  # -1 pour unknown
                    )
                    if self.debug:
                        unknown_count = (self.dataframe[f'{col}_encoded'] == -1).sum()
                        if unknown_count > 0:
                            print(f"Warning: {unknown_count} unknown labels in '{col}'")
                
                self.num_classes[col] = len(self.label_encoders[col].classes_)
        
        if self.debug:
            print(f"\nLabel columns: {self.label_cols}")
            print(f"Num classes per label: {self.num_classes}")
        
    def _setup_features(self, fit_encoders: bool):
        """Configure et encode les features temporelles et catégorielles."""
        from sklearn.preprocessing import LabelEncoder
        
        # === Encodage des features catégorielles ===
        self.categorical_dims = {}  # Dimensions pour embeddings
        
        for col in self.categorical_cols:
            if col not in self.dataframe.columns:
                continue
                
            if fit_encoders or col not in self.encoders:
                encoder = LabelEncoder()
                self.dataframe[f'{col}_encoded'] = encoder.fit_transform(
                    self.dataframe[col].fillna('unknown').astype(str)
                )
                self.encoders[col] = encoder
            else:
                # Utiliser l'encodeur existant (validation/test)
                encoder = self.encoders[col]
                # Gérer les valeurs inconnues
                known_classes = set(encoder.classes_)
                self.dataframe[f'{col}_encoded'] = self.dataframe[col].fillna('unknown').astype(str).apply(
                    lambda x: encoder.transform([x])[0] if x in known_classes else len(encoder.classes_)
                )
            
            self.categorical_dims[col] = len(self.encoders[col].classes_) + 1  # +1 pour unknown
        
        # === Features temporelles (normalisation) ===
        for col in self.temporal_cols:
            if col not in self.dataframe.columns:
                continue
            
            if fit_encoders:
                mean_val = self.dataframe[col].mean()
                std_val = self.dataframe[col].std() + 1e-8
                self.temporal_stats[col] = {'mean': mean_val, 'std': std_val}
            
            if col in self.temporal_stats:
                self.dataframe[f'{col}_norm'] = (
                    self.dataframe[col] - self.temporal_stats[col]['mean']
                ) / self.temporal_stats[col]['std']
        
        if self.debug:
            print(f"Categorical dims: {self.categorical_dims}")
            print(f"Temporal columns: {self.temporal_cols}")

    def get_class_weights(self, label_col: str = None) -> dict:
        """
        Calcule les poids de classe pour gérer le déséquilibre.
        
        Args:
            label_col: Colonne de label spécifique (ou tous si None)
            
        Returns:
            Dict avec les poids par classe pour chaque label
        """
        from sklearn.utils.class_weight import compute_class_weight
        
        weights = {}
        cols = [label_col] if label_col else self.label_cols
        
        for col in cols:
            enc_col = f'{col}_encoded'
            if enc_col not in self.dataframe.columns:
                continue
            y = self.dataframe[enc_col].values
            classes = np.unique(y[y >= 0])  # Ignorer les -1 (unknown)
            class_weights = compute_class_weight('balanced', classes=classes, y=y[y >= 0])
            weights[col] = torch.tensor(class_weights, dtype=torch.float32)
        
        return weights
    
    def get_label_distribution(self) -> dict:
        """Retourne la distribution des labels pour analyse."""
        distribution = {}
        for col in self.label_cols:
            enc_col = f'{col}_encoded'
            if enc_col in self.dataframe.columns:
                distribution[col] = self.dataframe[enc_col].value_counts(normalize=True).to_dict()
                # Ajouter le mapping si disponible
                if col in self.label_encoders:
                    distribution[f'{col}_mapping'] = dict(
                        zip(range(len(self.label_encoders[col].classes_)), 
                            self.label_encoders[col].classes_)
                    )
        return distribution
    
    def decode_labels(self, encoded_labels: np.ndarray, label_col: str) -> np.ndarray:
        """
        Décode les labels numériques en leurs valeurs originales (string).
        
        Args:
            encoded_labels: Array de labels encodés
            label_col: Nom de la colonne de label
            
        Returns:
            Array de labels décodés (strings)
        """
        if label_col in self.label_encoders:
            return self.label_encoders[label_col].inverse_transform(encoded_labels)
        return encoded_labels  # Déjà dans le format original

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        """
        Récupère un échantillon avec vidéo, labels et features auxiliaires.
        
        Returns:
            dict: {
                'video': Tensor [T, C, H, W],
                'labels': Tensor [num_labels] (encodés en int),
                'temporal_features': Tensor [num_temporal],
                'categorical_features': Tensor [num_categorical],
                'metadata': dict avec infos supplémentaires
            }
        """
        row = self.dataframe.iloc[idx]
        video_path = row['videos']
        
        # Chargement de la vidéo
        video = load_video(
            video_path,
            device=self.device,
            num_frames=self.num_frames,
            duration=self.duration,
            crop_duration=self.crop_duration,
            image_size=self.image_size,
            transform=self.transform,
            fallback_to_cpu=self.fallback_to_cpu,
        )
        
        # === Extraction des labels encodés (multi-label) ===
        labels = []
        for col in self.label_cols:
            enc_col = f'{col}_encoded'
            if enc_col in row:
                labels.append(int(row[enc_col]))
        labels = torch.tensor(labels, dtype=torch.long)
        
        # === Features temporelles normalisées ===
        temporal_features = []
        for col in self.temporal_cols:
            norm_col = f'{col}_norm'
            if norm_col in row:
                temporal_features.append(float(row[norm_col]))
        temporal_features = torch.tensor(temporal_features, dtype=torch.float32)
        
        # === Features catégorielles encodées ===
        categorical_features = []
        for col in self.categorical_cols:
            enc_col = f'{col}_encoded'
            if enc_col in row:
                categorical_features.append(int(row[enc_col]))
        categorical_features = torch.tensor(categorical_features, dtype=torch.long)
        
        output = {
            'video': video,
            'labels': labels,
            'temporal_features': temporal_features,
            'categorical_features': categorical_features,
            'metadata': {
                'video_path': video_path,
                'idx': idx,
            }
        }
        
        return output
    
    def get_feature_info(self) -> dict:
        """Retourne les informations sur les features pour initialiser le modèle."""
        return {
            'num_classes': self.num_classes,
            'label_cols': self.label_cols,
            'temporal_dim': len([c for c in self.temporal_cols if f'{c}_norm' in self.dataframe.columns]),
            'categorical_dims': self.categorical_dims,
            'encoders': self.encoders,
            'label_encoders': self.label_encoders,
            'temporal_stats': self.temporal_stats,
        }


In [ ]:
# =============================================================================
# CRÉATION DU DATASET AVEC MULTI-LABEL ET FEATURES AUXILIAIRES
# =============================================================================

# Créer le dataset avec gestion des features auxiliaires
ds = TrafficDataset(
    dataframe=df, 
    num_frames=CONFIG['NUM_FRAMES'], 
    duration=CONFIG['DURATION'], 
    crop_duration=CONFIG['CROP_DURATION'], 
    image_size=CONFIG['IMAGE_SIZE'], 
    device=CONFIG['DEVICE'], 
    transform=None,
    label_cols=['congestion_enter_rating', 'congestion_exit_rating'],
    temporal_cols=['time_segment_id'],
    categorical_cols=['signaling', 'cycle_phase'],
    fit_encoders=True,
    debug=True,
)

# Afficher les informations sur les features
feature_info = ds.get_feature_info()
print("\n" + "="*60)
print("Feature Information:")
print("="*60)
print(f"Labels: {feature_info['label_cols']}")
print(f"Num classes: {feature_info['num_classes']}")
print(f"Temporal dim: {feature_info['temporal_dim']}")
print(f"Categorical dims: {feature_info['categorical_dims']}")

# Afficher la distribution des labels (pour voir le déséquilibre)
print("\n" + "="*60)
print("Label Distribution (déséquilibre):")
print("="*60)
distribution = ds.get_label_distribution()
for label, dist in distribution.items():
    if 'mapping' in label:
        continue
    print(f"\n{label}:")
    for cls, ratio in sorted(dist.items()):
        print(f"  Class {cls}: {ratio:.2%}")

# Obtenir les poids de classe pour la loss pondérée
class_weights = ds.get_class_weights()
print("\n" + "="*60)
print("Class Weights (pour équilibrer la loss):")
print("="*60)
for label, weights in class_weights.items():
    print(f"{label}: {weights.numpy()}")
    
# DataLoader
dds = DataLoader(ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=0)

In [ ]:
def get_activation(activation: str) -> nn.Module:
    """
    Retourne le module d'activation correspondant au nom donné.
    
    Args:
        activation: Nom de la fonction d'activation ('relu', 'gelu', 'silu')
        
    Returns:
        nn.Module: Module d'activation PyTorch correspondant
        
    Raises:
        ValueError: Si le nom d'activation n'est pas reconnu
    """
    activations = {
        'relu': nn.ReLU,
        'gelu': nn.GELU,
        'silu': nn.SiLU,
    }
    if activation not in activations:
        raise ValueError(f"Unknown activation: {activation}. Choose from {list(activations.keys())}")
    return activations[activation]()
    
class VisionBackbone(nn.Module):
    """
    Backbone de vision basé sur un modèle pré-entraîné timm avec projection.
    
    Extrait des features visuelles de chaque frame d'une vidéo et les projette
    dans un espace de dimension inférieure.
    
    Attributes:
        backbone_name: Nom du modèle timm à utiliser
        projection_dim: Dimension de l'espace de projection
        pretrained: Utiliser les poids pré-entraînés
        _freeze_backbone: Flag pour geler le backbone
    """
    
    def __init__(
        self, 
        backbone_name: str, 
        projection_dim: int, 
        activation: str = 'relu', 
        pretrained: bool = True, 
        dropout: float = 0.0,
        freeze_backbone: bool = True,
        **kwargs
        ):
        """
        Initialise le VisionBackbone.
        
        Args:
            backbone_name: Nom du modèle timm (ex: 'efficientnet_b2')
            projection_dim: Dimension de sortie de la projection
            activation: Fonction d'activation ('relu', 'gelu', 'silu')
            pretrained: Si True, charge les poids pré-entraînés
            dropout: Taux de dropout dans la couche de projection
            freeze_backbone: Si True, gèle les poids du backbone
        """
        super().__init__(**kwargs)
        self.backbone_name = backbone_name
        self.projection_dim = projection_dim
        self.pretrained = pretrained    
        self._freeze_backbone_flag = freeze_backbone
        
        # Création du backbone avec timm
        self.vision_backbone = timm.create_model(
            model_name=backbone_name,  # Correction: 'arc' -> 'model_name'
            num_classes=0,  # Pas de tête de classification
            pretrained=pretrained,
        )
        
        self.vision_embedding_dim = self.vision_backbone.num_features

        # Couche de projection pour réduire la dimensionalité
        self.proj = nn.Sequential(
            nn.Linear(self.vision_embedding_dim, self.vision_embedding_dim // 2),
            get_activation(activation),
            nn.BatchNorm1d(self.vision_embedding_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(self.vision_embedding_dim // 2, projection_dim),
        )
        self.is_backbone_frozen = False
        
        # Geler le backbone si demandé
        if self._freeze_backbone_flag:
            self.freeze_backbone_weights()
        
    def freeze_backbone_weights(self):
        """Gèle les poids du backbone pour le transfer learning."""
        for param in self.vision_backbone.parameters():
            param.requires_grad = False
        self.vision_backbone.eval()
        self.is_backbone_frozen = True
    
    def unfreeze_backbone_weights(self):
        """Dégèle les poids du backbone (sauf BatchNorm) pour le fine-tuning."""
        for param in self.vision_backbone.parameters():
            param.requires_grad = True
            
        # Garder les BatchNorm gelés pour la stabilité
        for module in self.vision_backbone.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
                for param in module.parameters():
                    param.requires_grad = False
                
        self.vision_backbone.train()
        self.is_backbone_frozen = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass du backbone.
        
        Args:
            x: Tensor de forme [B, T, C, H, W] (batch, temps, canaux, hauteur, largeur)
            
        Returns:
            Tensor de forme [B, T, projection_dim] - embeddings projetés
        """
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)  # Flatten temporal dimension
        
        # Extraction des features
        if self.is_backbone_frozen:
            with torch.no_grad():
                features = self.vision_backbone(x)  # [B * T, D]
        else:   
            features = self.vision_backbone(x)  # [B * T, D]
            
        features = features.view(B, T, self.vision_embedding_dim)  # [B, T, D]
        
        # Projection - BatchNorm1d attend [B, C] ou [B, C, L]
        B, T, D = features.shape
        features = features.view(B * T, D)  # [B*T, D] pour BatchNorm1d
        embeddings = self.proj(features)  # [B*T, projection_dim]
        embeddings = embeddings.view(B, T, self.projection_dim)  # [B, T, projection_dim]
        
        return embeddings

## Modèle de Classification Vidéo

Architecture du modèle:
1. **VisionBackbone**: Extrait des features de chaque frame via un modèle pré-entraîné (EfficientNet, ResNet, etc.)
2. **TemporalEncoder**: LSTM bidirectionnel pour capturer les dépendances temporelles
3. **ClassificationHead**: Attention temporelle + MLP pour la classification finale

Le workflow d'entraînement suit le pattern Keras-like:
```python
model = VideoClassifier(...)
model.compile(optimizer, loss_fn, metrics)
model.fit(train_ds, val_ds, epochs)
```

In [ ]:
class TemporalEncoder(nn.Module):
    """
    Encodeur temporel basé sur LSTM bidirectionnel.
    
    Capture les dépendances temporelles dans une séquence de features visuelles.
    
    Attributes:
        input_dim: Dimension d'entrée
        hidden_dim: Dimension cachée du LSTM
        num_layers: Nombre de couches LSTM
        bidirectional: Si True, utilise un LSTM bidirectionnel
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout: float = 0.0,
        bidirectional: bool = True,
        **kwargs
        ):
        """
        Initialise le TemporalEncoder.
        
        Args:
            input_dim: Dimension des features d'entrée
            hidden_dim: Dimension cachée du LSTM
            num_layers: Nombre de couches LSTM empilées
            dropout: Taux de dropout entre les couches LSTM (ignoré si num_layers=1)
            bidirectional: Si True, utilise un LSTM bidirectionnel
        """
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.output_dim = hidden_dim * self.num_directions
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
            batch_first=True,
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass de l'encodeur temporel.
        
        Args:
            x: Tensor de forme [B, T, D] (batch, temps, dimension)
            
        Returns:
            Tensor de forme [B, T, H * num_directions] - features temporelles
        """
        outputs, _ = self.lstm(x)  # [B, T, H * num_directions]
        return outputs

In [ ]:
class MultiLabelClassificationHead(nn.Module):
    """
    Tête de classification multi-label avec attention temporelle et fusion de features auxiliaires.
    
    Supporte:
    - Multi-label output (congestion_enter_rating, congestion_exit_rating)
    - Features temporelles (time_segment_id, cycle_phase)
    - Features catégorielles avec embeddings (signaling)
    """
    
    def __init__(
        self,
        input_dim: int,
        num_classes_dict: dict,  # {'congestion_enter_rating': 5, 'congestion_exit_rating': 5}
        temporal_dim: int = 0,
        categorical_dims: dict = None,  # {'signaling': 10}
        embedding_dim: int = 16,
        dropout: float = 0.0,
        activation: str = 'relu',
        **kwargs
        ):
        """
        Args:
            input_dim: Dimension des features visuelles-temporelles
            num_classes_dict: Dict {label_name: num_classes} pour chaque sortie
            temporal_dim: Dimension des features temporelles continues
            categorical_dims: Dict {feature_name: num_categories} pour embeddings
            embedding_dim: Dimension des embeddings catégoriels
            dropout: Taux de dropout
            activation: Fonction d'activation
        """
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.num_classes_dict = num_classes_dict
        self.temporal_dim = temporal_dim
        self.categorical_dims = categorical_dims or {}
        self.embedding_dim = embedding_dim
        
        # Module de scoring pour l'attention temporelle
        self.scorer = nn.Sequential(
            nn.Conv1d(input_dim, input_dim, kernel_size=1, padding='same'),
            nn.BatchNorm1d(input_dim),
            nn.Softmax(dim=2),
        )
        
        # === Embeddings pour features catégorielles ===
        self.categorical_embeddings = nn.ModuleDict()
        total_cat_dim = 0
        for name, num_cat in self.categorical_dims.items():
            self.categorical_embeddings[name] = nn.Embedding(num_cat, embedding_dim)
            total_cat_dim += embedding_dim
        
        # Dimension totale après fusion
        fusion_dim = input_dim + temporal_dim + total_cat_dim
        
        # Couche de fusion
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, input_dim),
            get_activation(activation),
            nn.BatchNorm1d(input_dim),
            nn.Dropout(dropout),
        )
        
        # === Têtes de classification séparées pour chaque label ===
        self.classifiers = nn.ModuleDict()
        for label_name, num_classes in num_classes_dict.items():
            self.classifiers[label_name] = nn.Sequential(
                nn.Linear(input_dim, input_dim // 2),
                get_activation(activation),
                nn.BatchNorm1d(input_dim // 2),
                nn.Dropout(dropout),
                nn.Linear(input_dim // 2, num_classes),
            )
        
        self.label_names = list(num_classes_dict.keys())
        
    def forward(
        self, 
        x: torch.Tensor, 
        temporal_features: torch.Tensor = None,
        categorical_features: torch.Tensor = None,
        ) -> dict:
        """
        Forward pass.
        
        Args:
            x: Tensor [B, T, D] - features visuelles-temporelles
            temporal_features: Tensor [B, temporal_dim] - features temporelles
            categorical_features: Tensor [B, num_categorical] - indices catégoriels
            
        Returns:
            Dict {label_name: logits [B, num_classes]}
        """
        B = x.shape[0]
        
        # Attention temporelle sur les features vidéo
        x_t = x.transpose(1, 2)  # [B, D, T]
        scores = self.scorer(x_t)  # [B, D, T]
        pooled = (x_t * scores).sum(dim=2)  # [B, D]
        
        # Fusion des features
        features_to_concat = [pooled]
        
        # Ajouter features temporelles
        if temporal_features is not None and self.temporal_dim > 0:
            features_to_concat.append(temporal_features)
        
        # Ajouter embeddings catégoriels
        if categorical_features is not None and len(self.categorical_dims) > 0:
            cat_embeddings = []
            for i, name in enumerate(self.categorical_dims.keys()):
                cat_idx = categorical_features[:, i]
                emb = self.categorical_embeddings[name](cat_idx)
                cat_embeddings.append(emb)
            if cat_embeddings:
                features_to_concat.append(torch.cat(cat_embeddings, dim=1))
        
        # Fusion
        fused = torch.cat(features_to_concat, dim=1)
        fused = self.fusion(fused)
        
        # Classification multi-label
        outputs = {}
        for label_name in self.label_names:
            outputs[label_name] = self.classifiers[label_name](fused)
        
        return outputs

In [ ]:
class MetricTracker:
    """
    Utilitaire pour suivre et calculer les métriques pendant l'entraînement.
    
    Peut calculer des métriques à partir de prédictions/targets via une fonction,
    ou simplement accumuler des valeurs (comme la loss).
    
    Attributes:
        fn: Fonction de métrique optionnelle (prend preds, targets)
        name: Nom de la métrique
        values: Liste des valeurs accumulées
    """
    
    def __init__(self, fn=None, name: str = 'metric'):
        """
        Initialise le MetricTracker.
        
        Args:
            fn: Fonction de calcul de métrique (signature: fn(preds, targets) -> float)
            name: Nom descriptif de la métrique
        """
        self.fn = fn
        self.name = name
        self.reset()
    
    def reset(self):
        """Réinitialise les valeurs accumulées."""
        self.values = []
        
    def update(self, preds=None, targets=None, value=None):
        """
        Met à jour le tracker avec de nouvelles valeurs.
        
        Args:
            preds: Prédictions du modèle (Tensor ou array)
            targets: Labels cibles (Tensor ou array)
            value: Valeur directe à ajouter (utilisé si fn est None)
            
        Raises:
            ValueError: Si ni (fn + preds + targets) ni value ne sont fournis
        """
        # Conversion des tensors en numpy arrays
        if preds is not None and isinstance(preds, torch.Tensor):
            preds = preds.detach().cpu().numpy()
        if targets is not None and isinstance(targets, torch.Tensor):
            targets = targets.detach().cpu().numpy()    
        if value is not None and isinstance(value, torch.Tensor):
            value = value.detach().cpu().item()
            
        # Calcul de la métrique via la fonction
        if self.fn is not None and preds is not None and targets is not None:
            computed_value = self.fn(preds, targets)
            if isinstance(computed_value, (list, tuple, np.ndarray)):
                self.values.extend(list(computed_value))
            else:
                self.values.append(computed_value)
        # Ou ajout direct de la valeur
        elif value is not None:
            if isinstance(value, (list, tuple, np.ndarray)):
                self.values.extend(list(value))
            else:
                self.values.append(value)
        else:
            raise ValueError("Either fn with preds and targets, or value must be provided.")
    
    def compute(self) -> float:
        """
        Calcule la moyenne des valeurs accumulées.
        
        Returns:
            Moyenne des valeurs, ou 0.0 si aucune valeur
        """
        if len(self.values) == 0:
            return 0.0
        return sum(self.values) / len(self.values)
    
    def __repr__(self) -> str:
        return f"MetricTracker(name='{self.name}', num_values={len(self.values)})"

In [ ]:
def clear_memory():
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch._C._cuda_emptyCache()

# =============================================================================
# TRAINING CALLBACKS: ModelCheckpoint, EarlyStopping, LRScheduler, etc.
# =============================================================================

class Callback:
    """Base class for training callbacks."""
    
    def on_train_begin(self, model, logs=None): pass
    def on_train_end(self, model, logs=None): pass
    def on_epoch_begin(self, model, epoch, logs=None): pass
    def on_epoch_end(self, model, epoch, logs=None): pass
    def on_batch_begin(self, model, batch_idx, logs=None): pass
    def on_batch_end(self, model, batch_idx, logs=None): pass


class ModelCheckpoint(Callback):
    """
    Callback pour sauvegarder le meilleur modèle pendant l'entraînement.
    
    Args:
        filepath: Chemin de sauvegarde (supporte {epoch}, {val_loss}, etc.)
        monitor: Métrique à surveiller (ex: 'val_loss', 'val_accuracy')
        mode: 'min' (loss) ou 'max' (accuracy, f1, etc.)
        save_best_only: Si True, sauvegarde seulement le meilleur modèle
        save_weights_only: Si True, sauvegarde seulement les poids (pas l'optimizer)
        verbose: Niveau de verbosité
    """
    
    def __init__(
        self,
        filepath: str,
        monitor: str = 'val_loss',
        mode: str = 'min',
        save_best_only: bool = True,
        save_weights_only: bool = True,
        verbose: int = 1,
    ):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.save_best_only = save_best_only
        self.save_weights_only = save_weights_only
        self.verbose = verbose
        
        self.best_value = float('inf') if mode == 'min' else float('-inf')
        self.best_epoch = -1
        
    def _is_improvement(self, current: float) -> bool:
        if self.mode == 'min':
            return current < self.best_value
        return current > self.best_value
    
    def on_epoch_end(self, model, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        
        if current is None:
            if self.verbose > 0:
                print(f"⚠ ModelCheckpoint: '{self.monitor}' not found in logs. Available: {list(logs.keys())}")
            return
        
        filepath = self.filepath.format(epoch=epoch + 1, **logs)
        
        if self.save_best_only:
            if self._is_improvement(current):
                if self.verbose > 0:
                    print(f"✓ Epoch {epoch+1}: {self.monitor} improved from {self.best_value:.4f} to {current:.4f}")
                    print(f"  Saving model to {filepath}")
                self.best_value = current
                self.best_epoch = epoch
                self._save_model(model, filepath)
        else:
            if self.verbose > 0:
                print(f"  Saving model to {filepath}")
            self._save_model(model, filepath)
    
    def _save_model(self, model, filepath: str):
        os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)
        if self.save_weights_only:
            torch.save(model.state_dict(), filepath)
        else:
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': model.optimizer.state_dict() if model.optimizer else None,
                'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
            }, filepath)


class EarlyStopping(Callback):
    """
    Callback pour arrêter l'entraînement quand une métrique ne s'améliore plus.
    
    Args:
        monitor: Métrique à surveiller
        mode: 'min' ou 'max'
        patience: Nombre d'epochs sans amélioration avant arrêt
        min_delta: Amélioration minimale pour être considérée comme significative
        restore_best_weights: Si True, restaure les poids du meilleur epoch
        verbose: Niveau de verbosité
    """
    
    def __init__(
        self,
        monitor: str = 'val_loss',
        mode: str = 'min',
        patience: int = 5,
        min_delta: float = 0.0,
        restore_best_weights: bool = True,
        verbose: int = 1,
    ):
        super().__init__()
        self.monitor = monitor
        self.mode = mode
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.verbose = verbose
        
        self.best_value = float('inf') if mode == 'min' else float('-inf')
        self.best_weights = None
        self.best_epoch = -1
        self.wait = 0
        self.stopped_epoch = 0
        self.stop_training = False
        
    def _is_improvement(self, current: float) -> bool:
        if self.mode == 'min':
            return current < (self.best_value - self.min_delta)
        return current > (self.best_value + self.min_delta)
    
    def on_train_begin(self, model, logs=None):
        self.wait = 0
        self.stopped_epoch = 0
        self.stop_training = False
        self.best_value = float('inf') if self.mode == 'min' else float('-inf')
        self.best_weights = None
        
    def on_epoch_end(self, model, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        
        if current is None:
            if self.verbose > 0:
                print(f"⚠ EarlyStopping: '{self.monitor}' not found in logs.")
            return
        
        if self._is_improvement(current):
            self.best_value = current
            self.best_epoch = epoch
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.wait += 1
            if self.verbose > 0:
                print(f"  EarlyStopping: No improvement for {self.wait}/{self.patience} epochs")
            
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.stop_training = True
                if self.verbose > 0:
                    print(f"\n⛔ Early stopping triggered at epoch {epoch + 1}")
                    print(f"   Best {self.monitor}: {self.best_value:.4f} at epoch {self.best_epoch + 1}")
    
    def on_train_end(self, model, logs=None):
        if self.restore_best_weights and self.best_weights is not None:
            if self.verbose > 0:
                print(f"✓ Restoring best weights from epoch {self.best_epoch + 1}")
            model.load_state_dict(self.best_weights)


class ReduceLROnPlateau(Callback):
    """
    Callback pour réduire le learning rate quand une métrique stagne.
    
    Args:
        monitor: Métrique à surveiller
        mode: 'min' ou 'max'
        factor: Facteur de réduction (new_lr = lr * factor)
        patience: Epochs sans amélioration avant réduction
        min_lr: Learning rate minimum
        verbose: Niveau de verbosité
    """
    
    def __init__(
        self,
        monitor: str = 'val_loss',
        mode: str = 'min',
        factor: float = 0.5,
        patience: int = 3,
        min_lr: float = 1e-7,
        verbose: int = 1,
    ):
        super().__init__()
        self.monitor = monitor
        self.mode = mode
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.verbose = verbose
        
        self.best_value = float('inf') if mode == 'min' else float('-inf')
        self.wait = 0
        
    def _is_improvement(self, current: float) -> bool:
        if self.mode == 'min':
            return current < self.best_value
        return current > self.best_value
    
    def on_epoch_end(self, model, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        
        if current is None or model.optimizer is None:
            return
        
        if self._is_improvement(current):
            self.best_value = current
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.wait = 0
                for param_group in model.optimizer.param_groups:
                    old_lr = param_group['lr']
                    new_lr = max(old_lr * self.factor, self.min_lr)
                    if new_lr < old_lr:
                        param_group['lr'] = new_lr
                        if self.verbose > 0:
                            print(f"  📉 ReduceLROnPlateau: LR reduced from {old_lr:.2e} to {new_lr:.2e}")


class GradientClipping(Callback):
    """
    Callback pour le gradient clipping (évite l'explosion des gradients).
    
    Args:
        max_norm: Norme maximale du gradient
        norm_type: Type de norme (2 = L2 norm)
    """
    
    def __init__(self, max_norm: float = 1.0, norm_type: float = 2.0):
        super().__init__()
        self.max_norm = max_norm
        self.norm_type = norm_type
        
    def on_batch_end(self, model, batch_idx, logs=None):
        torch.nn.utils.clip_grad_norm_(model.parameters(), self.max_norm, self.norm_type)


class WarmupScheduler(Callback):
    """
    Callback pour warmup du learning rate au début de l'entraînement.
    
    Args:
        warmup_epochs: Nombre d'epochs de warmup
        initial_lr: LR de départ (0 = commence à 0)
        target_lr: LR cible après warmup
    """
    
    def __init__(
        self,
        warmup_epochs: int = 2,
        initial_lr: float = 1e-7,
        target_lr: float = 1e-3,
        verbose: int = 1,
    ):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.initial_lr = initial_lr
        self.target_lr = target_lr
        self.verbose = verbose
        
    def on_epoch_begin(self, model, epoch, logs=None):
        if epoch < self.warmup_epochs and model.optimizer is not None:
            # Linear warmup
            lr = self.initial_lr + (self.target_lr - self.initial_lr) * (epoch / self.warmup_epochs)
            for param_group in model.optimizer.param_groups:
                param_group['lr'] = lr
            if self.verbose > 0:
                print(f"  🔥 Warmup: LR set to {lr:.2e}")


class TrainingHistory(Callback):
    """
    Callback pour enregistrer l'historique des métriques.
    
    Attributes:
        history: Dict contenant les métriques par epoch
    """
    
    def __init__(self):
        super().__init__()
        self.history = {}
        
    def on_train_begin(self, model, logs=None):
        self.history = {}
        
    def on_epoch_end(self, model, epoch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            if key not in self.history:
                self.history[key] = []
            self.history[key].append(value)
    
    def plot(self, metrics: list = None, figsize=(12, 4)):
        """Affiche les courbes d'entraînement."""
        import matplotlib.pyplot as plt
        
        if metrics is None:
            # Trouver les paires train/val
            metrics = list(set(k.replace('train_', '').replace('val_', '') for k in self.history.keys()))
        
        n_metrics = len(metrics)
        fig, axes = plt.subplots(1, n_metrics, figsize=(figsize[0], figsize[1]))
        if n_metrics == 1:
            axes = [axes]
        
        for ax, metric in zip(axes, metrics):
            train_key = f'train_{metric}'
            val_key = f'val_{metric}'
            
            if train_key in self.history:
                ax.plot(self.history[train_key], label='Train', marker='o', markersize=3)
            if val_key in self.history:
                ax.plot(self.history[val_key], label='Val', marker='s', markersize=3)
            
            ax.set_xlabel('Epoch')
            ax.set_ylabel(metric)
            ax.set_title(metric.replace('_', ' ').title())
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        return fig


class CallbackList:
    """Container pour gérer plusieurs callbacks."""
    
    def __init__(self, callbacks: list = None):
        self.callbacks = callbacks or []
        
    def append(self, callback: Callback):
        self.callbacks.append(callback)
        
    def on_train_begin(self, model, logs=None):
        for cb in self.callbacks:
            cb.on_train_begin(model, logs)
            
    def on_train_end(self, model, logs=None):
        for cb in self.callbacks:
            cb.on_train_end(model, logs)
            
    def on_epoch_begin(self, model, epoch, logs=None):
        for cb in self.callbacks:
            cb.on_epoch_begin(model, epoch, logs)
            
    def on_epoch_end(self, model, epoch, logs=None):
        for cb in self.callbacks:
            cb.on_epoch_end(model, epoch, logs)
            
    def on_batch_begin(self, model, batch_idx, logs=None):
        for cb in self.callbacks:
            cb.on_batch_begin(model, batch_idx, logs)
            
    def on_batch_end(self, model, batch_idx, logs=None):
        for cb in self.callbacks:
            cb.on_batch_end(model, batch_idx, logs)
    
    def should_stop(self) -> bool:
        """Vérifie si un callback demande l'arrêt de l'entraînement."""
        for cb in self.callbacks:
            if hasattr(cb, 'stop_training') and cb.stop_training:
                return True
        return False
    
    def get_history(self) -> dict:
        """Récupère l'historique si TrainingHistory est présent."""
        for cb in self.callbacks:
            if isinstance(cb, TrainingHistory):
                return cb.history
        return {}

In [ ]:
from typing import Optional, Dict, Callable, Any, Union, List

class MultiLabelVideoClassifier(nn.Module):
    """
    Classifieur de vidéos multi-label avec features auxiliaires.
    
    Architecture:
    1. VisionBackbone: Extrait des features de chaque frame
    2. TemporalEncoder: Encode les dépendances temporelles via LSTM bidirectionnel
    3. MultiLabelClassificationHead: Classification multi-label avec fusion de features
    
    Supporte:
    - Multi-label: congestion_enter_rating, congestion_exit_rating
    - Features temporelles: time_segment_id, cycle_phase
    - Features catégorielles: signaling (avec embeddings)
    """
    
    def __init__(
        self,
        backbone_name: str,
        projection_dim: int,
        temporal_hidden_dim: int,
        temporal_num_layers: int,
        num_classes_dict: dict,  # {'congestion_enter_rating': 5, 'congestion_exit_rating': 5}
        temporal_feature_dim: int = 0,
        categorical_dims: dict = None,  # {'signaling': 10}
        embedding_dim: int = 16,
        activation: str = 'relu',
        pretrained: bool = True,
        dropout: float = 0.0,
        freeze_backbone: bool = True,
        device: str = 'auto',
        **kwargs
        ):
        """
        Args:
            backbone_name: Nom du modèle timm pour le backbone
            projection_dim: Dimension de projection des features visuelles
            temporal_hidden_dim: Dimension cachée du LSTM
            temporal_num_layers: Nombre de couches LSTM
            num_classes_dict: Dict {label_name: num_classes} pour chaque sortie
            temporal_feature_dim: Dimension des features temporelles (time_segment_id, cycle_phase)
            categorical_dims: Dict {feature_name: num_categories} pour embeddings
            embedding_dim: Dimension des embeddings catégoriels
            activation: Fonction d'activation
            pretrained: Utiliser les poids pré-entraînés pour le backbone
            dropout: Taux de dropout
            freeze_backbone: Geler le backbone pendant l'entraînement
            device: Device ('cuda', 'cpu', ou 'auto')
        """
        super().__init__(**kwargs)
        self.device = (
            device if device != 'auto' else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        )
        self.num_classes_dict = num_classes_dict
        self.label_names = list(num_classes_dict.keys())
        
        # Vision Backbone
        self.backbone = VisionBackbone(
            backbone_name=backbone_name,
            projection_dim=projection_dim,
            activation=activation,
            pretrained=pretrained,
            dropout=dropout,
            freeze_backbone=freeze_backbone,
        )
        
        # Temporal Encoder (LSTM bidirectionnel)
        self.temporal_encoder = TemporalEncoder(
            input_dim=projection_dim,
            hidden_dim=temporal_hidden_dim,
            num_layers=temporal_num_layers,
            dropout=dropout,
            bidirectional=True,
        )
        
        # Multi-Label Classification Head avec fusion
        self.classifier = MultiLabelClassificationHead(
            input_dim=temporal_hidden_dim * 2,  # *2 car bidirectionnel
            num_classes_dict=num_classes_dict,
            temporal_dim=temporal_feature_dim,
            categorical_dims=categorical_dims or {},
            embedding_dim=embedding_dim,
            dropout=dropout,
            activation=activation,
        )
        
        # Attributs pour l'entraînement
        self.loss_fns: Optional[Dict[str, nn.Module]] = None
        self.optimizer: Optional[torch.optim.Optimizer] = None
        self.scheduler: Optional[Any] = None
        self.metrics: Optional[Dict] = None
        self.class_weights: Optional[Dict[str, torch.Tensor]] = None
        self._is_compiled: bool = False
        
        self.to(self.device)
        
    @torch.autocast(device_type=CONFIG['DEVICE'], dtype=CONFIG['DTYPE'])
    def forward(
        self, 
        x: torch.Tensor,
        temporal_features: torch.Tensor = None,
        categorical_features: torch.Tensor = None,
        ) -> Dict[str, torch.Tensor]:
        """
        Forward pass du classifieur multi-label.
        
        Args:
            x: Tensor [B, T, C, H, W] - batch de vidéos
            temporal_features: Tensor [B, temporal_dim] - features temporelles
            categorical_features: Tensor [B, num_categorical] - indices catégoriels
            
        Returns:
            Dict {label_name: logits [B, num_classes]}
        """
        embeddings = self.backbone(x)  # [B, T, D]
        temporal_features_encoded = self.temporal_encoder(embeddings)  # [B, T, H*2]
        outputs = self.classifier(
            temporal_features_encoded, 
            temporal_features, 
            categorical_features
        )
        return outputs
    
    @torch.autocast(device_type=CONFIG['DEVICE'], dtype=CONFIG['DTYPE'])
    def compute_loss_fn(
        self, 
        outputs: Dict[str, torch.Tensor], 
        targets: torch.Tensor
        ) -> torch.Tensor:
        """
        Calcule la loss totale (somme des losses pour chaque label).
        
        Args:
            outputs: Dict {label_name: logits}
            targets: Tensor [B, num_labels] - labels pour chaque sortie
            
        Returns:
            Total loss
        """
        total_loss = 0.0
        for i, label_name in enumerate(self.label_names):
            label_targets = targets[:, i]
            logits = outputs[label_name]
            loss = self.loss_fns[label_name](logits, label_targets)
            total_loss = total_loss + loss
        return total_loss / len(self.label_names)
    
    @torch.inference_mode()
    def predict(
        self, 
        x: torch.Tensor,
        temporal_features: torch.Tensor = None,
        categorical_features: torch.Tensor = None,
        ) -> Dict[str, torch.Tensor]:
        """Prédit les probabilités pour chaque label."""
        self.eval()
        outputs = self(x, temporal_features, categorical_features)
        probs = {name: torch.softmax(logits, dim=1) for name, logits in outputs.items()}
        return probs
    
    def _set_metric_tracker(self, metrics: Optional[Dict] = None):
        """Configure les trackers de métriques pour chaque label."""
        # Loss trackers par label
        self.loss_trackers = {
            label: MetricTracker(name=f'{label}_loss', fn=None) 
            for label in self.label_names
        }
        self.total_loss_tracker = MetricTracker(name='total_loss', fn=None)
        
        # Metric trackers par label
        self.metric_trackers = {}
        if metrics:
            for metric_name, metric_fn in metrics.items():
                for label in self.label_names:
                    tracker_name = f'{label}_{metric_name}'
                    self.metric_trackers[tracker_name] = MetricTracker(fn=metric_fn, name=tracker_name)
    
    def train_step(self, batch: Dict, batch_idx: int, return_loss: bool = True):
        """Effectue une étape d'entraînement."""
        self.train()
        
        videos = batch['video'].to(self.device)
        labels = batch['labels'].to(self.device)
        temporal_features = batch.get('temporal_features')
        categorical_features = batch.get('categorical_features')
        
        if temporal_features is not None:
            temporal_features = temporal_features.to(self.device)
        if categorical_features is not None:
            categorical_features = categorical_features.to(self.device)
        
        self.optimizer.zero_grad()
        outputs = self(videos, temporal_features, categorical_features)
        loss = self.compute_loss_fn(outputs, labels)
        loss.backward()
        self.optimizer.step()
        
        if self.scheduler is not None:
            self.scheduler.step()
            
        return (outputs, loss) if return_loss else outputs
    
    @torch.inference_mode()
    def eval_step(self, batch: Dict, batch_idx: int):
        """Effectue une étape d'évaluation."""
        self.eval()
        
        videos = batch['video'].to(self.device)
        labels = batch['labels'].to(self.device)
        temporal_features = batch.get('temporal_features')
        categorical_features = batch.get('categorical_features')
        
        if temporal_features is not None:
            temporal_features = temporal_features.to(self.device)
        if categorical_features is not None:
            categorical_features = categorical_features.to(self.device)
        
        outputs = self(videos, temporal_features, categorical_features)
        loss = self.compute_loss_fn(outputs, labels)
        return outputs, loss
    
    def compile(
        self, 
        optimizer: type,
        loss_fn: nn.Module = None,
        metrics: Optional[Dict[str, Callable]] = None,
        scheduler: Optional[Any] = None, 
        class_weights: Optional[Dict[str, torch.Tensor]] = None,
        optimizer_kwargs: Optional[Dict] = None,
        ):
        """
        Configure le modèle pour l'entraînement.
        
        Args:
            optimizer: Classe d'optimiseur
            loss_fn: Fonction de loss (appliquée à tous les labels)
            metrics: Dict de métriques {nom: fonction}
            scheduler: Scheduler de learning rate
            class_weights: Dict {label_name: weights tensor} pour le déséquilibre
            optimizer_kwargs: Arguments pour l'optimiseur
        """
        if optimizer_kwargs is None:
            optimizer_kwargs = {'lr': 1e-3, 'weight_decay': 1e-5}
        
        # Configurer les poids de classe et loss pour chaque label
        self.class_weights = class_weights or {}
        self.loss_fns = {}
        
        for label_name, num_classes in self.num_classes_dict.items():
            if label_name in self.class_weights:
                weights = self.class_weights[label_name].to(self.device)
                self.loss_fns[label_name] = loss_fn(weight=weights)
            else:
                self.loss_fns[label_name] = loss_fn()
        
        # Optimiseur
        self.optimizer = optimizer(self.parameters(), **optimizer_kwargs)
        
        # Scheduler
        self.scheduler = scheduler
        
        # Métriques
        self._set_metric_tracker(metrics)
        self._is_compiled = True
        
        print(f"✓ Model compiled successfully")
        print(f"  - Optimizer: {optimizer.__name__}")
        print(f"  - Labels: {self.label_names}")
        print(f"  - Class weights: {list(self.class_weights.keys()) if self.class_weights else 'None'}")
        print(f"  - Metrics: {list(metrics.keys()) if metrics else 'None'}")
        
    def _metric_to_string(self, metrics: Dict[str, float]) -> str:
        """Convertit un dict de métriques en string formaté."""
        return ', '.join([f"{k}: {v:.4f}" for k, v in metrics.items()])

    @torch.inference_mode()
    def compute_metrics(
        self, 
        outputs: Dict[str, torch.Tensor], 
        targets: torch.Tensor, 
        loss: Optional[torch.Tensor] = None, 
        phase: str = 'train'
        ) -> Dict[str, float]:
        """Calcule toutes les métriques configurées pour chaque label."""
        metrics = {}
        
        # Loss totale
        if loss is not None:
            self.total_loss_tracker.update(value=loss)
            metrics[f"{phase}_loss"] = self.total_loss_tracker.compute()
        
        # Métriques par label
        for i, label_name in enumerate(self.label_names):
            label_targets = targets[:, i]
            logits = outputs[label_name]
            
            for tracker_name, tracker in self.metric_trackers.items():
                if tracker_name.startswith(f'{label_name}_'):
                    tracker.update(preds=logits, targets=label_targets)
        
        for name, tracker in self.metric_trackers.items():
            metrics[f"{phase}_{name}"] = tracker.compute()
            
        return metrics
    
    def reset_metrics(self):
        """Réinitialise tous les trackers de métriques."""
        self.total_loss_tracker.reset()
        for tracker in self.loss_trackers.values():
            tracker.reset()
        for tracker in self.metric_trackers.values():
            tracker.reset()
        
    def fit(
        self, 
        train_ds: DataLoader, 
        val_ds: Optional[DataLoader] = None, 
        epochs: int = 1, 
        verbose: int = 1,
        callbacks: list = None,
        ):
        """
        Entraîne le modèle avec support complet des callbacks.
        
        Args:
            train_ds: DataLoader d'entraînement
            val_ds: DataLoader de validation (optionnel)
            epochs: Nombre d'epochs
            verbose: Niveau de verbosité (0=silencieux, 1=barre de progression)
            callbacks: Liste de callbacks (ModelCheckpoint, EarlyStopping, etc.)
        
        Returns:
            dict: Historique des métriques si TrainingHistory callback est présent
        """
        if not self._is_compiled:
            raise RuntimeError("Model must be compiled before training. Call 'compile()' first.")
        
        # === Setup callbacks ===
        callback_list = CallbackList(callbacks or [])
        
        # Ajouter TrainingHistory si non présent
        has_history = any(isinstance(cb, TrainingHistory) for cb in callback_list.callbacks)
        if not has_history:
            history_callback = TrainingHistory()
            callback_list.append(history_callback)
        
        print("=" * 60)
        print("Starting training...")
        print(f"  Device: {self.device}")
        print(f"  Epochs: {epochs}")
        print(f"  Training samples: {len(train_ds.dataset)}")
        if val_ds is not None:
            print(f"  Validation samples: {len(val_ds.dataset)}")
        print(f"  Callbacks: {[type(cb).__name__ for cb in callback_list.callbacks]}")
        print("=" * 60)
        
        # Callback: on_train_begin
        callback_list.on_train_begin(self)
        
        disable_pbar = verbose == 0
        pbar = tqdm(range(epochs), desc='Epochs', unit='epoch', leave=True, disable=disable_pbar)
        
        for epoch in pbar:
            # Callback: on_epoch_begin
            callback_list.on_epoch_begin(self, epoch)
            
            self.reset_metrics()
            train_metrics = {}
            val_metrics = {}
            train_metrics_str = ""
            val_metrics_str = ""
            
            # === Boucle d'entraînement ===
            self.train()
            inner_pbar = tqdm(train_ds, desc='Training', leave=False, unit='batch', disable=disable_pbar)
            for batch_idx, batch in enumerate(inner_pbar):
                # Callback: on_batch_begin
                callback_list.on_batch_begin(self, batch_idx)
                
                outputs, loss = self.train_step(batch, batch_idx, return_loss=True)
                train_metrics = self.compute_metrics(outputs, batch['labels'], loss, phase='train')
                train_metrics_str = self._metric_to_string(train_metrics)
                inner_pbar.set_description(f"Training - {train_metrics_str}")
                
                # Callback: on_batch_end
                callback_list.on_batch_end(self, batch_idx, logs=train_metrics)
                
                del outputs, loss
                clear_memory()
            
            # === Boucle de validation ===
            if val_ds is not None:
                self.reset_metrics()
                self.eval()
                val_pbar = tqdm(val_ds, desc='Validation', leave=False, ncols=100, disable=disable_pbar)
                for batch_idx, batch in enumerate(val_pbar):
                    outputs, loss = self.eval_step(batch, batch_idx)
                    val_metrics = self.compute_metrics(outputs, batch['labels'], loss, phase='val')
                    val_metrics_str = self._metric_to_string(val_metrics)
                    del outputs, loss
                    clear_memory()
                pbar.set_description(f"Epoch {epoch+1}/{epochs} - {train_metrics_str} - {val_metrics_str}")
            else:
                pbar.set_description(f"Epoch {epoch+1}/{epochs} - {train_metrics_str}")
            
            # Combine all metrics for callbacks
            all_metrics = {**train_metrics, **val_metrics}
            
            # Callback: on_epoch_end
            callback_list.on_epoch_end(self, epoch, logs=all_metrics)
            
            # Check early stopping
            if callback_list.should_stop():
                print(f"\n🛑 Training stopped at epoch {epoch + 1}")
                break
        
        # Callback: on_train_end
        callback_list.on_train_end(self)
        
        print("=" * 60)
        print("Training completed!")
        print("=" * 60)
        
        return callback_list.get_history()
        
    def save(self, path: str):
        """Sauvegarde les poids du modèle."""
        torch.save(self.state_dict(), path)
        print(f"✓ Model saved to {path}")
        
    def load(self, path: str):
        """Charge les poids du modèle."""
        self.load_state_dict(torch.load(path, map_location=self.device))
        print(f"✓ Model loaded from {path}")

In [ ]:
# =============================================================================
# CRÉATION DU MODÈLE MULTI-LABEL
# =============================================================================

# Récupérer les informations du dataset
feature_info = ds.get_feature_info()

# Créer le modèle multi-label
model = MultiLabelVideoClassifier(
    backbone_name='efficientnet_b2',
    projection_dim=256,
    temporal_hidden_dim=128,
    temporal_num_layers=2,
    num_classes_dict=feature_info['num_classes'],  # {'congestion_enter_rating': N, 'congestion_exit_rating': M}
    temporal_feature_dim=feature_info['temporal_dim'],  # time_segment_id, cycle_phase
    categorical_dims=feature_info['categorical_dims'],  # {'signaling': K}
    embedding_dim=16,
    activation='relu',
    pretrained=True,
    dropout=0.3,  # Augmenté pour le déséquilibre
    freeze_backbone=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

# Afficher le résumé du modèle
print(f"\n{'='*60}")
print("Model Summary")
print('='*60)
print(f"Device: {model.device}")
print(f"Labels: {model.label_names}")
print(f"Num classes per label: {model.num_classes_dict}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print('='*60)

In [ ]:
# =============================================================================
# MÉTRIQUES POUR CLASSIFICATION DÉSÉQUILIBRÉE
# =============================================================================
# Pour un problème de classification déséquilibrée, les métriques importantes sont:
# 1. Weighted F1-Score: Pondère chaque classe par son support
# 2. Macro F1-Score: Moyenne non pondérée (sensible aux classes minoritaires)
# 3. Balanced Accuracy: Moyenne des rappels par classe
# 4. Quadratic Weighted Kappa (QWK): Mesure l'accord inter-annotateur (utile pour ordinal)
# 5. Per-class metrics: Pour identifier les classes problématiques

from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    balanced_accuracy_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
)

def compute_accuracy(preds, targets):
    """Accuracy standard."""
    pred_classes = np.argmax(preds, axis=1)
    return accuracy_score(targets, pred_classes)

def compute_balanced_accuracy(preds, targets):
    """Balanced Accuracy - moyenne des rappels par classe (robuste au déséquilibre)."""
    pred_classes = np.argmax(preds, axis=1)
    return balanced_accuracy_score(targets, pred_classes)

def compute_f1_weighted(preds, targets):
    """Weighted F1-Score - pondéré par le support de chaque classe."""
    pred_classes = np.argmax(preds, axis=1)
    return f1_score(targets, pred_classes, average='weighted', zero_division=0)

def compute_f1_macro(preds, targets):
    """Macro F1-Score - moyenne non pondérée (sensible aux classes minoritaires)."""
    pred_classes = np.argmax(preds, axis=1)
    return f1_score(targets, pred_classes, average='macro', zero_division=0)

def compute_qwk(preds, targets):
    """
    Quadratic Weighted Kappa (QWK).
    Idéal pour les problèmes de classification ordinale (rating 1-5).
    Pénalise plus fortement les erreurs éloignées de la vraie classe.
    """
    pred_classes = np.argmax(preds, axis=1)
    return cohen_kappa_score(targets, pred_classes, weights='quadratic')

def compute_linear_kappa(preds, targets):
    """Linear Weighted Kappa - pénalise linéairement les erreurs."""
    pred_classes = np.argmax(preds, axis=1)
    return cohen_kappa_score(targets, pred_classes, weights='linear')

# Fonction utilitaire pour afficher le rapport de classification complet
def print_classification_report(preds, targets, label_name: str = ""):
    """Affiche un rapport de classification détaillé."""
    pred_classes = np.argmax(preds, axis=1)
    print(f"\n{'='*60}")
    print(f"Classification Report: {label_name}")
    print('='*60)
    print(classification_report(targets, pred_classes, zero_division=0))
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(targets, pred_classes))
    print('='*60)

In [ ]:
# =============================================================================
# COMPILATION ET ENTRAÎNEMENT DU MODÈLE
# =============================================================================

# Obtenir les poids de classe du dataset
class_weights = ds.get_class_weights()

# Compiler le modèle avec les métriques pour classification déséquilibrée
model.compile(
    optimizer=torch.optim.AdamW,
    loss_fn=nn.CrossEntropyLoss,  # Sera remplacé par weighted CE pour chaque label
    metrics={
        'accuracy': compute_accuracy,
        'balanced_acc': compute_balanced_accuracy,
        'f1_weighted': compute_f1_weighted,
        'f1_macro': compute_f1_macro,
        'qwk': compute_qwk,  # Quadratic Weighted Kappa (important pour rating ordinal)
    },
    class_weights=class_weights,  # Poids pour gérer le déséquilibre
    optimizer_kwargs={
        'lr': 1e-3,
        'weight_decay': 1e-4,
    }
)

print("\n" + "="*60)
print("Métriques utilisées pour la classification déséquilibrée:")
print("="*60)
print("- accuracy: Accuracy standard (biaisée vers classe majoritaire)")
print("- balanced_acc: Moyenne des rappels par classe (non biaisée)")
print("- f1_weighted: F1 pondéré par support (balance precision/rappel)")
print("- f1_macro: F1 macro (sensible aux classes minoritaires)")
print("- qwk: Quadratic Weighted Kappa (idéal pour rating ordinal)")
print("="*60)

In [ ]:
# =============================================================================
# TRAIN/VALIDATION SPLIT ET ENTRAÎNEMENT
# =============================================================================

# Split stratifié pour maintenir la distribution des classes
from sklearn.model_selection import train_test_split
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# ===== CRÉER LE TRANSFORM POUR LE BACKBONE =====
# Résoudre la config du modèle pour obtenir les bonnes normalisations
backbone_model = timm.create_model('efficientnet_b2', pretrained=False)
data_config = resolve_data_config(backbone_model.pretrained_cfg)
print("Data config from backbone:")
print(f"  - mean: {data_config['mean']}")
print(f"  - std: {data_config['std']}")
print(f"  - input_size: {data_config['input_size']}")

# Créer le transform (sans augmentation pour l'inférence/validation)
# Le transform convertit automatiquement en float et normalise
inference_transform = create_transform(
    input_size=data_config['input_size'],
    mean=data_config['mean'],
    std=data_config['std'],
    is_training=False,  # Pas d'augmentation
)

# Transform pour l'entraînement avec augmentation légère
train_transform = create_transform(
    input_size=data_config['input_size'],
    mean=data_config['mean'],
    std=data_config['std'],
    is_training=True,  # Avec augmentation
    auto_augment=None,  # Désactiver auto augment pour vidéo
    re_prob=0.0,  # Pas de random erasing
    re_mode='const',
)

print(f"\nInference transform: {inference_transform}")
print(f"\nTrain transform: {train_transform}")

# Split sur un des labels (ou combinaison)
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['congestion_enter_rating'],  # Stratifier sur un label
    random_state=42
)

print(f"\nTrain size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

# Créer les datasets train/val AVEC les transforms
train_ds = TrafficDataset(
    dataframe=train_df, 
    num_frames=CONFIG['NUM_FRAMES'], 
    duration=CONFIG['DURATION'], 
    crop_duration=CONFIG['CROP_DURATION'], 
    image_size=CONFIG['IMAGE_SIZE'], 
    device=CONFIG['DEVICE'], 
    transform=inference_transform,  # Utiliser le transform (inference pour stabilité)
    label_cols=['congestion_enter_rating', 'congestion_exit_rating'],
    temporal_cols=['time_segment_id'],
    categorical_cols=['signaling', 'cycle_phase'],
    fit_encoders=True,
    debug=True,
)

# Val dataset utilise les encodeurs du train (labels + features)
feature_info_train = train_ds.get_feature_info()
val_ds = TrafficDataset(
    dataframe=val_df, 
    num_frames=CONFIG['NUM_FRAMES'], 
    duration=CONFIG['DURATION'], 
    crop_duration=CONFIG['CROP_DURATION'], 
    image_size=CONFIG['IMAGE_SIZE'], 
    device=CONFIG['DEVICE'], 
    transform=inference_transform,  # Même transform pour validation
    label_cols=['congestion_enter_rating', 'congestion_exit_rating'],
    temporal_cols=['time_segment_id'],
    categorical_cols=['signaling', 'cycle_phase'],
    fit_encoders=False,
    encoders=feature_info_train['encoders'],
    label_encoders=feature_info_train['label_encoders'],
    temporal_stats=feature_info_train['temporal_stats'],
    debug=True,
)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=0)

# Afficher le mapping des labels
print("\n" + "="*60)
print("Label Mapping (string -> int):")
print("="*60)
for col, encoder in train_ds.label_encoders.items():
    print(f"\n{col}:")
    for i, cls in enumerate(encoder.classes_):
        print(f"  '{cls}' -> {i}")

# Obtenir les poids de classe du train set
class_weights = train_ds.get_class_weights()

# Créer le modèle avec les bonnes dimensions
feature_info = train_ds.get_feature_info()
model = MultiLabelVideoClassifier(
    backbone_name='efficientnet_b2',
    projection_dim=256,
    temporal_hidden_dim=128,
    temporal_num_layers=2,
    num_classes_dict=feature_info['num_classes'],
    temporal_feature_dim=feature_info['temporal_dim'],
    categorical_dims=feature_info['categorical_dims'],
    embedding_dim=16,
    activation='relu',
    pretrained=True,
    dropout=0.3,
    freeze_backbone=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

# Compiler avec les poids du train set
model.compile(
    optimizer=torch.optim.AdamW,
    loss_fn=nn.CrossEntropyLoss,
    metrics={
        'accuracy': compute_accuracy,
        'balanced_acc': compute_balanced_accuracy,
        'f1_weighted': compute_f1_weighted,
        'qwk': compute_qwk,
    },
    class_weights=class_weights,
    optimizer_kwargs={
        'lr': 1e-4,
        'weight_decay': 1e-4,
    }
)

In [ ]:
# =============================================================================
# LANCER L'ENTRAÎNEMENT AVEC CALLBACKS
# =============================================================================

# Définir les callbacks essentiels pour un entraînement robuste
callbacks = [
    # Sauvegarder le meilleur modèle basé sur la balanced accuracy de validation
    ModelCheckpoint(
        filepath='/kaggle/working/checkpoints/best_model_epoch{epoch}.pt',
        monitor='val_congestion_enter_rating_balanced_acc',  # ou 'val_loss'
        mode='max',  # 'max' pour accuracy, 'min' pour loss
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    
    # Early stopping si pas d'amélioration après N epochs
    EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=5,  # Arrêter après 5 epochs sans amélioration
        min_delta=0.001,  # Amélioration minimale significative
        restore_best_weights=True,  # Restaurer les meilleurs poids à la fin
        verbose=1,
    ),
    
    # Réduire le LR si la métrique stagne
    ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        factor=0.5,  # Diviser LR par 2
        patience=3,  # Après 3 epochs sans amélioration
        min_lr=1e-7,
        verbose=1,
    ),
    
    # Gradient clipping pour stabilité
    GradientClipping(max_norm=1.0),
    
    # Warmup du learning rate
    WarmupScheduler(
        warmup_epochs=2,
        initial_lr=1e-7,
        target_lr=1e-4,
        verbose=1,
    ),
    
    # Historique pour visualisation
    TrainingHistory(),
]

# Créer le dossier de checkpoints
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

# Entraîner le modèle avec callbacks
history = model.fit(
    train_ds=train_loader,
    val_ds=val_loader,
    epochs=20,  # Plus d'epochs car EarlyStopping arrêtera si besoin
    verbose=1,
    callbacks=callbacks,
)

# Sauvegarder le modèle final
model.save('/kaggle/working/traffic_classifier_multilabel_final.pt')

# Afficher les courbes d'entraînement
print("\n" + "="*60)
print("Training History")
print("="*60)
for cb in callbacks:
    if isinstance(cb, TrainingHistory):
        cb.plot(metrics=['loss', 'congestion_enter_rating_balanced_acc'])
        break